# 07 · CUDA utilization

Pairs with `GUIDE.md` 9-10 and `docs/04` modules M0-M6. The measurement cells need a
GPU; on CPU they print a note. Run this on your **RTX 3060 (Ampere, sm_86, 28 SMs)**.

In [ ]:
import torch
from gpulab.learn import inspect as I
HAS_CUDA = torch.cuda.is_available()
dev = torch.device("cuda" if HAS_CUDA else "cpu")
print("device:", dev, "| CUDA:", HAS_CUDA)
if not HAS_CUDA:
    print("No CUDA here - cells run on CPU; the timing/memory numbers are only")
    print("meaningful on your RTX 3060. Run this notebook there for the real story.")

## M0 — device inventory & ceilings (derive, don't look up)

In [ ]:
from gpulab.train.cuda_utils import describe_device
print(describe_device())
if HAS_CUDA:
    p = torch.cuda.get_device_properties(0)
    cores = p.multi_processor_count * 128        # Ampere: 128 FP32 lanes / SM
    print("SMs:", p.multi_processor_count, "| CUDA cores ~", cores,
          "| L2:", getattr(p, "l2_cache_size", "n/a"))
    # Your turn: from cores * 2 * boost_clock estimate FP32 TFLOPS; from bus width x
    # mem clock estimate GB/s; divide -> the FLOP/byte balance point. # TODO

## Whole dataset on the GPU

In [ ]:
import numpy as np
X = torch.tensor(np.random.default_rng(0).standard_normal((200_000, 120)).astype("float32"))
I.reset_cuda_peak()
Xg = X.to(dev)
if HAS_CUDA: I.cuda_mem("200k curves resident")
print("dataset MB:", round(X.numel()*4/1e6, 1))   # fits easily in 12 GB

## M2/M5 — launch-bound vs compute-bound

In [ ]:
from gpulab.models.cnn1d import CNN1D
model = CNN1D(n_classes=3, roi_len=120).to(dev)
step = lambda: model(Xg[:512])
if HAS_CUDA:
    I.cuda_time(step, iters=100)     # prints gpu vs cpu ms -> LAUNCH-bound?
else:
    print("run on GPU: I.cuda_time reports launch- vs compute-bound")

## M6 — batch-size sweep (find the knee)

In [ ]:
if HAS_CUDA:
    for bs in (64, 256, 1024, 4096, 16384):
        n = min(bs, Xg.shape[0])
        ms = I.cuda_time(lambda: model(Xg[:n]), iters=30)
        print(f"batch {bs:6d}: {ms:.3f} ms  -> {n/ (ms/1e3):.0f} samples/s")
    print("Watch `nvidia-smi dmon -s um` in another terminal; note where throughput flattens.")
else:
    print("GPU-only: throughput vs batch, and where it saturates the 28 SMs.")

## M4/G10 — mixed precision

In [ ]:
I.autocast_probe(device=str(dev), dtype=torch.bfloat16)   # Ampere supports bf16
# Your turn (GPU): wrap model(Xg[:4096]) in torch.autocast and compare I.cuda_time
# and I.cuda_mem with vs without AMP. # TODO

> **Concepts to note** (copy into your own theory notebook):
> - Whole dataset fits in VRAM -> no DataLoader; slice batches on-device.
> - Tiny model + tiny curves => LAUNCH-bound: GPU idle waiting on kernel dispatch.
> - Throughput vs batch has a knee where SMs finally fill; below it the GPU is starved.
> - Ampere has bf16 + TF32 (unlike Turing/T4). bf16 needs no gradient scaler.
> - `sm%` in nvidia-smi = "a kernel was resident", not "SMs were full".